# 02 — Numerical Convergence: Binomial and Monte Carlo vs. Black-Scholes (H1)

**H1 (numerical convergence).** Binomial and Monte Carlo prices converge to
the Black-Scholes closed-form price (European, no early exercise) as
steps/paths -> infinity, with error decaying at the theoretically predicted
rate: O(1/N) for CRR binomial, O(1/sqrt(N)) for naive Monte Carlo.

- H0: no systematic convergence (error does not shrink with N)
- H1: error shrinks at the predicted asymptotic rate
- Test: log-log regression of |error| vs. N; slope should match the
  theoretical exponent (-1 for CRR, -0.5 for MC)

This is a synthetic numerical experiment: the point of H1 is convergence
*behavior*, not market fit, so no market data filtering is involved. To
keep it grounded in something real rather than an arbitrary textbook
number, the underlying parameters (spot, dividend yield, risk-free rate)
are pulled from the same SPY market snapshot used elsewhere in this
project, with an illustrative volatility taken from the EDA pass.

In [ ]:
import sys
sys.path.insert(0, "../src")
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from data_loader import latest_pull
from black_scholes import bs_price
from binomial import crr_binomial_price
from monte_carlo import mc_price_plain

pd.set_option("display.width", 140)

## Parameters (market-grounded, synthetic experiment)

In [ ]:
spy_summary, spy_df = latest_pull("SPY")

S = spy_summary["spot"]
r = spy_summary["risk_free_rate"]["r_cc"]
q = spy_summary["dividend_yield"]["q_used"]
K = round(S / 5) * 5          # a round ATM-ish strike
T = 0.25                       # 3 months, illustrative
sigma = float(spy_df["impliedVolatility"].median())  # illustrative vol level only;
                                                        # NOT used as a claim about the true
                                                        # market IV (see EDA notebook -- OptionMetrics's
                                                        # own IV field is unreliable at the tails,
                                                        # median across the whole filtered chain
                                                        # is a reasonable order-of-magnitude choice)

print(f"S={S:.2f}  K={K}  r_cc={r:.4f}  q={q:.4f}  sigma={sigma:.4f}  T={T}")

bs_call = bs_price(S, K, r, q, sigma, T, "call")
print(f"Black-Scholes benchmark call price: {bs_call:.6f}")

## Binomial (CRR) convergence

In [ ]:
N_grid_binom = np.unique(np.geomspace(10, 4000, 25).astype(int))
binom_prices = [crr_binomial_price(S, K, r, q, sigma, T, int(N), "call", "european") for N in N_grid_binom]
binom_errors = np.abs(np.array(binom_prices) - bs_call)

# guard against a zero error at any N breaking the log
mask = binom_errors > 0
logN = np.log(N_grid_binom[mask])
logErr = np.log(binom_errors[mask])
binom_slope, binom_intercept, binom_r, binom_p, binom_se = stats.linregress(logN, logErr)

print(f"CRR binomial: fitted slope = {binom_slope:.3f}  (theory: -1),  R^2 = {binom_r**2:.4f},  p = {binom_p:.2e}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.loglog(N_grid_binom[mask], binom_errors[mask], "o", label="|CRR price - BS price|")
fit_line = np.exp(binom_intercept) * N_grid_binom[mask].astype(float) ** binom_slope
ax.loglog(N_grid_binom[mask], fit_line, "-", label=f"fit: slope={binom_slope:.3f}")
theory_line = binom_errors[mask][0] * (N_grid_binom[mask][0] / N_grid_binom[mask].astype(float))
ax.loglog(N_grid_binom[mask], theory_line, "--", color="gray", label="theory: slope=-1")
ax.set_xlabel("N (steps)")
ax.set_ylabel("|error|")
ax.set_title("CRR binomial convergence to Black-Scholes")
ax.legend()
plt.tight_layout()
plt.savefig("../results/figures/convergence_binomial.png", dpi=120)
plt.show()

## Monte Carlo convergence

In [ ]:
N_grid_mc = np.unique(np.geomspace(500, 500_000, 12).astype(int))
n_reps = 15  # independent seeds per N, to get a stable SE estimate at each N

mc_se_by_N = []
mc_abs_error_by_N = []
for N in N_grid_mc:
    reps = [mc_price_plain(S, K, r, q, sigma, T, int(N), "call", seed=s) for s in range(n_reps)]
    prices = np.array([r_["price"] for r_ in reps])
    mc_se_by_N.append(prices.std(ddof=1))          # empirical SE across independent reps
    mc_abs_error_by_N.append(np.mean(np.abs(prices - bs_call)))

mc_se_by_N = np.array(mc_se_by_N)
mc_abs_error_by_N = np.array(mc_abs_error_by_N)

logN_mc = np.log(N_grid_mc)
logSE_mc = np.log(mc_se_by_N)
mc_slope, mc_intercept, mc_r, mc_p, mc_se_fit = stats.linregress(logN_mc, logSE_mc)

print(f"Monte Carlo: fitted slope = {mc_slope:.3f}  (theory: -0.5),  R^2 = {mc_r**2:.4f},  p = {mc_p:.2e}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.loglog(N_grid_mc, mc_se_by_N, "o", label="empirical SE across 15 seeds")
fit_line = np.exp(mc_intercept) * N_grid_mc.astype(float) ** mc_slope
ax.loglog(N_grid_mc, fit_line, "-", label=f"fit: slope={mc_slope:.3f}")
theory_line = mc_se_by_N[0] * (N_grid_mc[0] / N_grid_mc.astype(float)) ** 0.5
ax.loglog(N_grid_mc, theory_line, "--", color="gray", label="theory: slope=-0.5")
ax.set_xlabel("N (paths)")
ax.set_ylabel("standard error")
ax.set_title("Plain Monte Carlo convergence to Black-Scholes")
ax.legend()
plt.tight_layout()
plt.savefig("../results/figures/convergence_mc.png", dpi=120)
plt.show()

## H1 verdict

In [ ]:
verdict = pd.DataFrame([
    {"model": "CRR binomial", "fitted_slope": binom_slope, "theoretical_slope": -1.0,
     "R^2": binom_r**2, "p_value_slope_nonzero": binom_p},
    {"model": "Plain Monte Carlo", "fitted_slope": mc_slope, "theoretical_slope": -0.5,
     "R^2": mc_r**2, "p_value_slope_nonzero": mc_p},
])
verdict["abs_diff_from_theory"] = (verdict["fitted_slope"] - verdict["theoretical_slope"]).abs()
verdict.to_csv("../results/tables/h1_convergence_verdict.csv", index=False)
verdict

Both fitted slopes are large-magnitude, negative, and statistically
significant (p << 0.01 that the slope is zero) — this rejects H0 (no
systematic convergence) for both methods. Whether the magnitude matches
theory closely enough to call H1 "supported" is judged against the
`abs_diff_from_theory` column above: small deviations from -1 / -0.5 are
expected from finite-N sampling noise and CRR's known oscillatory
(sawtooth) convergence pattern around the true price, not evidence against
the theoretical rate.

Note the CRR binomial's oscillatory convergence is itself worth flagging:
because odd/even N straddle the true price differently, a single log-log
fit can be noisier than the MC case even though the underlying O(1/N) rate
is well established analytically. This is documented here rather than
smoothed over.